# 面试题：Agent 间怎样用 Artifact Handoff 交接？

交接的是带 schema、hash、来源、权限和版本的 artifact，不是自由文本总结。接收者先验证 hash、schema、租户与有效期，再消费；hash 只能说明未篡改，不能代替来源和内容验收。下面演示研究 Agent 向采购 Agent 交接报价表。

## 真实案例

六个报价 artifact 含合法对象、hash 篡改、租户错误、过期、schema 升级和未验收对象。

## 基线

基线只读取摘要文本中的最低价。

## 结果解读

手写校验逐项输出 hash、ACL、版本与验收结论。

## 失败案例

摘要看似正常，但 artifact hash 与内容不一致时必须拒绝。

In [1]:
artifacts = [{'id':'A1','content':'报价:ThinkPad:800','hash':'20','actual':'20','tenant':'t1','schema':1,'accepted':True,'expired':False}, {'id':'A2','content':'报价:Mac:900','hash':'19','actual':'20','tenant':'t1','schema':1,'accepted':True,'expired':False}, {'id':'A3','content':'报价:HP:700','hash':'18','actual':'18','tenant':'t2','schema':1,'accepted':True,'expired':False}, {'id':'A4','content':'报价:Dell:750','hash':'21','actual':'21','tenant':'t1','schema':1,'accepted':True,'expired':True}, {'id':'A5','content':'报价:Lenovo:780','hash':'23','actual':'23','tenant':'t1','schema':2,'accepted':True,'expired':False}, {'id':'A6','content':'报价:Asus:650','hash':'18','actual':'18','tenant':'t1','schema':1,'accepted':False,'expired':False}]  # 构造六个带简化 hash、租户和验收状态的交接对象。
print('Artifact 输入:', artifacts)  # 输出交接对象的完整元数据。
print('教学说明：hash 用字符串长度模拟，生产应使用密码学 hash 与不可变存储。')  # 明确教学简化。

Artifact 输入: [{'id': 'A1', 'content': '报价:ThinkPad:800', 'hash': '20', 'actual': '20', 'tenant': 't1', 'schema': 1, 'accepted': True, 'expired': False}, {'id': 'A2', 'content': '报价:Mac:900', 'hash': '19', 'actual': '20', 'tenant': 't1', 'schema': 1, 'accepted': True, 'expired': False}, {'id': 'A3', 'content': '报价:HP:700', 'hash': '18', 'actual': '18', 'tenant': 't2', 'schema': 1, 'accepted': True, 'expired': False}, {'id': 'A4', 'content': '报价:Dell:750', 'hash': '21', 'actual': '21', 'tenant': 't1', 'schema': 1, 'accepted': True, 'expired': True}, {'id': 'A5', 'content': '报价:Lenovo:780', 'hash': '23', 'actual': '23', 'tenant': 't1', 'schema': 2, 'accepted': True, 'expired': False}, {'id': 'A6', 'content': '报价:Asus:650', 'hash': '18', 'actual': '18', 'tenant': 't1', 'schema': 1, 'accepted': False, 'expired': False}]
教学说明：hash 用字符串长度模拟，生产应使用密码学 hash 与不可变存储。


In [2]:
baseline = [(item['id'], item['content'].split(':')[-1]) for item in artifacts]  # 构造只解析摘要末尾价格的基线。
print('摘要基线:', baseline)  # 输出不检查来源与完整性的错误消费。
print('基线问题：篡改、跨租户和过期 artifact 都会被当成可信输入。')  # 指出文本交接风险。

摘要基线: [('A1', '800'), ('A2', '900'), ('A3', '700'), ('A4', '750'), ('A5', '780'), ('A6', '650')]
基线问题：篡改、跨租户和过期 artifact 都会被当成可信输入。


In [3]:
def consume(item, tenant='t1', schema=1):  # 定义接收 Agent 的 artifact 消费门禁。
    checks = {'hash':item['hash'] == item['actual'],'tenant':item['tenant'] == tenant,'schema':item['schema'] == schema,'accepted':item['accepted'],'fresh':not item['expired']}  # 计算五项可审计校验。
    return ('accepted' if all(checks.values()) else 'rejected'), checks  # 只有所有契约均满足才允许进入后续计划。

In [4]:
results = [(item['id'],) + consume(item) for item in artifacts]  # 对六个交接对象执行消费校验。
print('id | 交接结论 | 校验项')  # 输出 artifact handoff 结果表标题。
for item in results:  # 遍历每个 artifact 的接受或拒绝证据。
    print(item[0], item[1], item[2])  # 输出哈希、租户、schema、验收和时效中间量。
print('可消费数:', sum(status == 'accepted' for _, status, _ in results))  # 汇总真正可传给采购 Agent 的对象数。

id | 交接结论 | 校验项
A1 accepted {'hash': True, 'tenant': True, 'schema': True, 'accepted': True, 'fresh': True}
A2 rejected {'hash': False, 'tenant': True, 'schema': True, 'accepted': True, 'fresh': True}
A3 rejected {'hash': True, 'tenant': False, 'schema': True, 'accepted': True, 'fresh': True}
A4 rejected {'hash': True, 'tenant': True, 'schema': True, 'accepted': True, 'fresh': False}
A5 rejected {'hash': True, 'tenant': True, 'schema': False, 'accepted': True, 'fresh': True}
A6 rejected {'hash': True, 'tenant': True, 'schema': True, 'accepted': False, 'fresh': True}
可消费数: 1


In [5]:
wrong = dict(baseline)['A2']  # 读取篡改 artifact 在摘要基线中的价格。
fixed = dict((item[0], item[1]) for item in results)['A2']  # 读取 hash 校验后的拒绝结论。
print('失败案例 A2：摘要价格=', wrong, '，交接校验=', fixed)  # 展示 hash 不一致不能继续消费。
print('生产差距：需要密码学 hash、签名来源、schema 兼容转换、最小权限下载、TTL 与 artifact 审计。')  # 说明真实交接实现。

失败案例 A2：摘要价格= 900 ，交接校验= rejected
生产差距：需要密码学 hash、签名来源、schema 兼容转换、最小权限下载、TTL 与 artifact 审计。


In [6]:
assert dict((item[0], item[1]) for item in results)['A1'] == 'accepted'  # 验证完整合法 artifact 可消费。
assert dict((item[0], item[1]) for item in results)['A2'] == 'rejected'  # 验证内容篡改会被拒绝。
assert dict((item[0], item[1]) for item in results)['A3'] == 'rejected'  # 验证跨租户 artifact 不会泄露。